# L9c: Logistic Regression and Numerical Optimization
In this lecture, we'll explore logistic regression, a fundamental technique for binary classification. Unlike the Perceptron's deterministic approach, we now adopt a probabilistic framework for binary classification. This allows us to not only predict class labels but also quantify confidence in our predictions. We'll also develop the numerical optimization methods we use to estimate the model parameters.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Derive the logistic regression model for binary classification:__ Logistic regression models the probability that an example belongs to each class as a function of its features. We'll derive the logistic function from the Boltzmann distribution and use the resulting probabilities to assign class labels.
> * __Estimate logistic regression parameters with gradient descent:__ We choose the parameters that make the observed training labels most likely, which is the same as minimizing the cross-entropy loss. We'll derive the gradient of this loss, minimize it with gradient descent, and add a penalty on the size of the parameters (regularization).
> * __Solve constrained problems with simulated annealing:__ Simulated annealing is a derivative-free optimization method that sometimes accepts a worse candidate solution so it can escape a local minimum. We'll handle constraints with penalty and barrier terms and work through the algorithm and the choice of its temperature schedule.

We'll demonstrate these ideas with an example, where we fit a logistic regression classifier to synthetic data with gradient descent and with simulated annealing. Let's get started!
___

## Examples
We will use the following example to illustrate the key ideas of this lecture:

> [▶ Fit a logistic regression classifier with gradient descent and simulated annealing](CHEME-5800-L9c-Example-LogisticRegression-GD-SA-Fall-2026.ipynb). How do the two optimization methods in this lecture compare on the same problem? We generate two overlapping clusters of labeled points, fit a logistic regression model with gradient descent, and use its probabilities to classify test points at several thresholds. We then fit the same model with simulated annealing, which never computes a gradient, and compare the two answers.

___

## Logistic Regression
In [the L9a lecture](../L9a/CHEME-5800-L9a-Lecture-LinearClassificationAndPerceptron-Fall-2026.ipynb), the Perceptron labeled an example $y\in\{-1,1\}$ by the sign of the score $\hat{\mathbf{x}}^{\top}\mathbf{\theta}$, where the augmented features $\hat{\mathbf{x}}^{\top}=(x_{1},\dots,x_{m},1)$ hold the $m$ features and a `1` for the bias, and $\mathbf{\theta}=(w_{1},\dots,w_{m},b)\in\mathbb{R}^{p}$ holds the weights and the bias, with $p=m+1$. The set where the score equals zero is the __decision boundary__. __Logistic regression__ keeps the Perceptron's linear score but replaces the sign function with a probability that rises smoothly with the score:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-Logistic-Function/Fig-Logistic-Function.svg" width="620" style="max-width:100%; height:auto;" alt="Plot of the probability of the label y equals plus 1 against the score x-hat transpose theta, from minus 6 to 6. Three S-shaped curves for beta equal to one quarter (blue, gentle), one (black), and four (red, nearly a step) all pass through one half at a score of zero, where a dashed vertical line marks the decision boundary. A dashed horizontal line marks probability one half. Labels read predict y-hat equals minus 1 on the left and predict y-hat equals plus 1 on the right."/>
</p>

__The logistic function.__ Each curve shows the probability of the label $y=+1$ against the score $\hat{\mathbf{x}}^{\top}\mathbf{\theta}$ for one value of the inverse temperature $\beta$, a parameter of the model. Every curve crosses $1/2$ at the decision boundary, and a larger $\beta$ makes the rise sharper.

Let's derive this function, starting from the Boltzmann distribution.

### Derivation
Suppose we view our two-class labels $y\in\{-1,1\}$ as _states_ in a [Boltzmann distribution](https://en.wikipedia.org/wiki/Boltzmann_distribution) conditioned on the input $\hat{\mathbf{x}}$, in which lower-energy states are more probable. For any state $y$ with energy $E(y,\hat{\mathbf{x}})$ at inverse temperature $\beta=1/T>0$, where $T$ is the temperature, the conditional probability of observing the label $y$ given the feature vector $\hat{\mathbf{x}}$ is given by:
$$
\begin{align*}
P(y\mid \hat{\mathbf{x}})
=\frac{\exp\bigl(-\beta E(y,\hat{\mathbf{x}})\bigr)}
      {\underbrace{\sum_{y' \in\{-1,1\}} \exp\bigl(-\beta E(y',\hat{\mathbf{x}})\bigr)}_{Z(\hat{\mathbf{x}})}}
\end{align*}
$$
The partition function $Z(\hat{\mathbf{x}})$ normalizes the probabilities so they sum to one across both labels. For the energy function, we use a linear model of the form:
$$
\begin{align*}
E(y,\hat{\mathbf{x}})\;=\;-\,y\;\bigl(\hat{\mathbf{x}}^{\top}\mathbf{\theta} \bigr)
\end{align*}
$$
where $\mathbf{\theta}\in\mathbb{R}^{p}$ is a vector of __unknown__ parameters (weights plus bias) that we want to learn. When $y=+1$, the energy $E(1,\hat{\mathbf{x}})=-\hat{\mathbf{x}}^{\top}\mathbf{\theta}$ is *lower* (more probable) if $\hat{\mathbf{x}}^{\top}\mathbf{\theta}$ is large. On the other hand, when $y=-1$, the energy $E(-1,\hat{\mathbf{x}})=+\hat{\mathbf{x}}^{\top}\mathbf{\theta}$, so $y=-1$ is favored when $\hat{\mathbf{x}}^{\top}\mathbf{\theta}$ is very negative.

Let's substitute the energy function into the conditional probability expression and do some algebra:
$$
\begin{align*}
P_{\mathbf{\theta}}(y\mid \hat{\mathbf{x}})
& =\frac{\exp\bigl(-\beta E(y,\hat{\mathbf{x}})\bigr)}
      {\underbrace{\sum_{y' \in\{-1,1\}} \exp\bigl(-\beta E(y',\hat{\mathbf{x}})\bigr)}_{Z(\hat{\mathbf{x}})}}\\
&=\frac{\exp\bigl(\beta y\left(\hat{\mathbf{x}}^{\top}\mathbf{\theta}\right)\bigr)}
      {\exp\bigl(\beta\hat{\mathbf{x}}^{\top}\mathbf{\theta}\bigr) + \exp\bigl(-\beta\hat{\mathbf{x}}^{\top}\mathbf{\theta}\bigr)}\quad\Longrightarrow\;{\text{substituting } z = \beta\hat{\mathbf{x}}^{\top}\mathbf{\theta}}\\
& = \frac{\exp\bigl(yz\bigr)}
      {\exp\bigl(z\bigr) + \exp\bigl(-z\bigr)}\quad\Longrightarrow\;{\text{factor out}\; \exp(yz)\;\text{from denominator}}\\
& = \frac{\exp\bigl(yz\bigr)}
      {\exp\bigl(yz\bigr)\left(\exp\bigl((1-y)z\bigr) + \exp\bigl(-(1+y)z\bigr)\right)}\quad\Longrightarrow\;\text{cancel}\;\exp(yz)\\
& = \frac{1}
      {\exp\bigl((1-y)z\bigr) + \exp\bigl(-(1+y)z\bigr)}\quad\blacksquare
\end{align*}
$$

This expression is the probability of observing the label $y$ given the feature vector $\hat{\mathbf{x}}$ and the parameters $\mathbf{\theta}$. Let's look at the cases $y=+1$ and $y=-1$:

> __Cases:__
>
> When $y=+1$, we have:
>
> $$
> \begin{align*}
> P_{\mathbf{\theta}}(y = +1\mid \hat{\mathbf{x}})
> & = \frac{1}{\exp\bigl(0\bigr) + \exp\bigl(-2\beta\left(\hat{\mathbf{x}}^{\top}\mathbf{\theta}\right)\bigr)}\\
> & = \frac{1}{1 + \exp\bigl(-2\beta\left(\hat{\mathbf{x}}^{\top}\mathbf{\theta}\right)\bigr)}\quad\blacksquare
> \end{align*}
> $$
>
> When $y=-1$, we have:
>
> $$
> \begin{align*}
> P_{\mathbf{\theta}}(y = -1\mid \hat{\mathbf{x}})
> & = \frac{1}{\exp\bigl(2\beta\left(\hat{\mathbf{x}}^{\top}\mathbf{\theta}\right)\bigr) + \exp\bigl(0\bigr)}\\
> & = \frac{1}{1+\exp\bigl(2\beta\left(\hat{\mathbf{x}}^{\top}\mathbf{\theta}\right)\bigr)}\quad\blacksquare
> \end{align*}
> $$
>
> Putting this all together, we can write the conditional probability of observing the label $y$ given the feature vector $\hat{\mathbf{x}}$ and the parameters $\mathbf{\theta}$ as:
>
> $$
> \begin{align*}
> P_{\mathbf{\theta}}(y\mid \hat{\mathbf{x}}) & = \frac{1}{1+\exp\bigl(-2\beta y\left(\hat{\mathbf{x}}^{\top}\mathbf{\theta}\right)\bigr)}\quad\Longrightarrow\;\text{Logistic function!}\\
> & = \sigma\bigl(2\beta y\left(\hat{\mathbf{x}}^{\top}\mathbf{\theta}\right)\bigr)
> \end{align*}
> $$
>
> where $\sigma(z) = 1/\left(1+e^{-z}\right)$ is the [logistic function](https://en.wikipedia.org/wiki/Logistic_function).

The logistic function maps any real-valued input to the interval $(0, 1)$, making it ideal for modeling probabilities. Because $\sigma(-z)=1-\sigma(z)$, the probabilities of the two labels sum to one.

### Understanding the Inverse Temperature $\beta$
The inverse temperature $\beta > 0$ controls how the energy translates into probabilities. Think of $\beta$ as controlling the "confidence" of the model in its decisions, as the figure shows:

> __Physical intuition for $\beta$:__
>
> * __Large $\beta$ (cold system, low temperature):__ Small energy differences translate into large probability differences, and the model becomes highly confident. As $\beta \to \infty$, the smooth logistic function becomes a hard threshold, resembling the Perceptron's sign function.
> * __Small $\beta$ (hot system, high temperature):__ All states become roughly equally probable regardless of their energy. As $\beta \to 0$, the probability of each label approaches $1/2$ for every input.

__In practice__, $\beta$ appears only in the product $\beta\,\mathbf{\theta}$, so we fix $\beta$ and learn $\mathbf{\theta}$; under maximum likelihood, a different $\beta$ only rescales the estimated parameters. Setting $\beta=1/2$ gives the common form $P_{\mathbf{\theta}}(y\mid\hat{\mathbf{x}})=\sigma(y\,\hat{\mathbf{x}}^{\top}\mathbf{\theta})$.

### From Probabilities to Class Labels
To classify an example, we predict the more probable label: $\hat{y}=+1$ when $P_{\mathbf{\theta}}(y=+1\mid\hat{\mathbf{x}})\geq 1/2$ and $\hat{y}=-1$ otherwise. Because $\sigma(z)\geq 1/2$ exactly when $z\geq 0$, this is the Perceptron's rule $\hat{y}=\texttt{sign}(\hat{\mathbf{x}}^{\top}\mathbf{\theta})$, with the same decision boundary.

__Choosing a threshold.__ We can instead predict $\hat{y}=+1$ when the probability is at least some other threshold $t\in(0,1)$. Lowering $t$ labels more examples positive: the recall (the fraction of actual positives the classifier finds) can only rise or stay the same, and so can the number of false positives (negative examples labeled positive). When missing a positive is costly, as in disease screening, a threshold below $1/2$ may be the better choice.

___

## Parameter Estimation: Learning from Data
Now that we have a probabilistic model, we need to learn the parameters $\mathbf{\theta}$ that make our model fit the observed data well, here a dataset $\mathcal{D} = \left\{(\mathbf{x}_{i},y_{i}) \mid i = 1,2,\dots,n\right\}$ of $n$ labeled examples. We do this by finding parameters that maximize the likelihood of observing the training labels given the features. Assuming the training examples are independent, the likelihood function is given by:
$$
\begin{align*}
\mathcal{L}(\mathbf{\theta}) & = \prod_{i=1}^{n} P_{\mathbf{\theta}}(y_{i}\mid \hat{\mathbf{x}}_{i})\\
& = \prod_{i=1}^{n} \frac{1}{1+\exp\bigl(-2\beta y_{i}\,\left(\hat{\mathbf{x}}^{\top}_{i}\mathbf{\theta}\right)\bigr)}\quad\Longrightarrow\;\text{Product is $\textbf{hard}$ to optimize! Take the $\log$}\\
\log\mathcal{L}(\mathbf{\theta}) & = -\sum_{i=1}^n \log\!\bigl(1+\exp\bigl(-2\beta y_i\,\left(\hat{\mathbf{x}}^{\top}_{i}\mathbf{\theta}\right)\bigr)\bigr)
\end{align*}
$$
We can use gradient descent to minimize the negative log-likelihood (also known as the __cross-entropy loss__ function):
$$
\boxed{
\begin{align*}
J(\mathbf{\theta}) & = -\log\mathcal{L}(\mathbf{\theta})\\
& = \sum_{i=1}^n \log\!\bigl(1+\exp\bigl(-2\beta y_i\,\left(\hat{\mathbf{x}}^{\top}_{i}\mathbf{\theta}\right)\bigr)\bigr)\quad\blacksquare
\end{align*}}
$$
This will give us the optimal parameters $\mathbf{\theta}$ for our logistic regression model:
$$
\begin{equation*}
\hat{\mathbf{\theta}} = \arg\min_{\mathbf{\theta}} J(\mathbf{\theta})
\end{equation*}
$$
Unlike least squares, setting the gradient to zero gives no equation we can solve for $\hat{\mathbf{\theta}}$ directly, so we minimize $J(\mathbf{\theta})$ numerically.

### Gradient Descent
<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-GD-Schematic/Fig-GD-Schematic.svg" width="680" style="max-width:100%; height:auto;" alt="Plot of an objective (loss) function against a single parameter theta. The black curve rises steeply on the left, dips to a local minimum at theta 2, rises over a hump, falls to a lower minimum at theta 1, and rises again on the right. A blue arrow at the start point theta 3 on the left points down the slope toward theta 2. A green arrow at the start point theta 0 on the right points down the slope toward theta 1. Dashed lines mark the loss values f of theta 1 and f of theta 2 on the vertical axis."/>
</p>

__Gradient descent and local minima.__ Gradient descent minimizes a function by iteratively adjusting the parameters in the _opposite direction_ of the gradient. Started at $\theta_{0}$, it goes downhill to the lowest point, $\theta_{1}$. Started at $\theta_{3}$, it stops at $\theta_{2}$, a __local minimum__: no nearby point is lower, but $\theta_{1}$ is lower still.

> __Gradient?__ The gradient is a vector that points in the direction of the steepest increase of the function, and its magnitude indicates how steep the slope is in that direction. Thus, the negative gradient points in the direction of the steepest decrease of the function.

For the cross-entropy loss, we can compute the gradient by hand:

> __Gradient of the cross-entropy loss:__ Let $u_{i} = 2\beta y_{i}\,\hat{\mathbf{x}}_{i}^{\top}\mathbf{\theta}$, so that term $i$ of the loss is $\log\left(1+e^{-u_{i}}\right)$. By the chain rule:
>
> $$
> \begin{align*}
> \nabla_{\mathbf{\theta}}\log\left(1+e^{-u_{i}}\right) = \left(\frac{-e^{-u_{i}}}{1+e^{-u_{i}}}\right)\nabla_{\mathbf{\theta}}u_{i} = -\bigl(1-\sigma(u_{i})\bigr)\,2\beta y_{i}\,\hat{\mathbf{x}}_{i}
> \end{align*}
> $$
>
> Summing over the training examples gives the gradient of the loss:
>
> $$
> \boxed{
> \begin{align*}
> \nabla J(\mathbf{\theta}) = -2\beta\sum_{i=1}^{n}\bigl(1-\sigma(u_{i})\bigr)\,y_{i}\,\hat{\mathbf{x}}_{i}\quad\blacksquare
> \end{align*}}
> $$

The factor $1-\sigma(u_{i})$ is the probability the model gives to the _wrong_ label for example $i$, so each gradient descent step moves $\mathbf{\theta}$ toward $y_{i}\,\hat{\mathbf{x}}_{i}$ for every example, weighted by this probability. The Perceptron update $\mathbf{\theta}\gets\mathbf{\theta}+y\,\hat{\mathbf{x}}$ uses only the mistakes, each with full weight; logistic regression uses every example, but those it already classifies correctly and confidently contribute almost nothing.

### Gradient Descent Algorithm
Let's examine a simple algorithm for estimating the parameters $\mathbf{\theta}$ using gradient descent.

__Initialization:__ We start with an initial guess for the parameters $\mathbf{\theta}^{(0)}\in\mathbb{R}^{p}$, which can be a random vector or a vector of zeros. Specify a learning rate $\alpha(k)>0$ (which can be constant or a function of the iteration count $k$) and a stopping criterion, e.g., a small threshold $\epsilon>0$ and a maximum number of iterations $\texttt{maxiter}$. Set $\texttt{converged}\gets\texttt{false}$ and the iteration count $k\gets0$.

While not $\texttt{converged}$ __do__:
1. Compute the update direction $\mathbf{d} = -\nabla J(\mathbf{\theta}^{(k)})$, the negative gradient of the loss at the current parameters.
2. Update the parameters $\mathbf{\theta}^{(k+1)} = \mathbf{\theta}^{(k)} + \alpha(k)\cdot\mathbf{d}$.
3. Check the stopping criterion:
   - If $\lVert\mathbf{\theta}^{(k+1)} - \mathbf{\theta}^{(k)}\rVert_{2}\leq\epsilon$ __then__ set $\texttt{converged}\gets\texttt{true}$ and return $\mathbf{\theta}^{(k+1)}$. Alternatively, we can stop when the gradient is small, i.e., $\lVert\nabla J(\mathbf{\theta}^{(k)})\rVert_{2}\leq\epsilon$.
   - If $k+1\geq\texttt{maxiter}$ __then__ return $\mathbf{\theta}^{(k+1)}$ and warn that the maximum number of iterations was reached _without convergence_.
4. Increment the iteration count $k\gets k+1$, and update the learning rate $\alpha(k)$, if applicable.

Let's take a closer look at two components of this algorithm:
* __What is $\alpha(k)$?__ The (hyper) parameter $\alpha(k)>0$ is the _learning rate_, a user-adjustable parameter that can be a function of the iteration count $k$; we'll assume it's constant for today. The learning rate controls how much we adjust the parameters at each iteration. If it's too large, we may overshoot the minimum; if it's too small, convergence may be slow.
* __Stopping?__ Here, we have a choice in defining the stopping criterion, such as which quantity to monitor or which norm to use. The choice of stopping criterion can affect when the algorithm stops and how close the result is to the minimum.

### How Good Is Gradient Descent?
Like most things in life, it depends. It depends on the problem, the learning rate you choose, and your stopping rule. In practice, gradient descent works very well for logistic regression, thanks to the convex shape of the loss:

> __Convexity:__ A function $f$ is convex if for any two inputs $\mathbf{a}, \mathbf{b}$ and any weight $t \in [0,1]$:
>
> $$
> f\bigl(t\,\mathbf{a} + (1-t)\,\mathbf{b}\bigr) \leq t\,f(\mathbf{a}) + (1-t)\,f(\mathbf{b})
> $$
>
> Geometrically, the line segment connecting any two points on the graph lies on or above the graph. A convex function has no local minimum like $\theta_{2}$ in the figure: any local minimum is a global minimum. For example, $f(x) = x^2$ and $f(x) = e^x$ are convex, while $f(x) = -x^2$ is not.

Each term of the loss is a convex function of $u_{i}$, which is linear in $\mathbf{\theta}$, so $J(\mathbf{\theta})$ is convex. What does this mean for gradient descent?
* __A smooth, convex loss:__ The gradient can't change too abruptly, so if you pick a learning rate up to $\alpha=1/L$, where $L$ is the Lipschitz constant of the gradient, every step lowers the loss unless the gradient is already zero:
    $$
    \lVert\nabla J(\mathbf{a})-\nabla J(\mathbf{b})\rVert_{2}\leq L\,\lVert\mathbf{a}-\mathbf{b}\rVert_{2},\qquad L=\beta^{2}\lVert\hat{\mathbf{X}}\rVert_{2}^{2}
    $$
    where $\lVert\hat{\mathbf{X}}\rVert_{2}$ is the largest singular value of the data matrix $\hat{\mathbf{X}}$, whose rows are the augmented features $\hat{\mathbf{x}}_{i}^{\top}$.
* __Slowing down over time:__ Initially, you make large improvements in the loss, but each subsequent step helps a bit less than the last. When the loss has a minimizer, your gap to the optimal loss after $k$ steps shrinks on the order of $1/k$. In other words, you sprint at first, then settle into a steady jog.
* __When there is no minimum:__ If the training data are linearly separable, scaling up a separating $\mathbf{\theta}$ makes every term of the loss smaller, so the loss has no minimizer and gradient descent keeps growing $\mathbf{\theta}$. Regularization fixes this.

Faster variants of gradient descent, such as momentum methods and Newton's method, are described in the [advanced optimization notebook](CHEME-5800-L9c-Advanced-NonlinearOptimization-Fall-2026.ipynb).

__Alternatives to gradient descent?__ Yes, computing the gradient is a _hassle_. There are alternatives that evaluate only the objective function, such as the [Nelder-Mead Simplex Algorithm](https://en.wikipedia.org/wiki/Nelder%E2%80%93Mead_method), [Genetic Algorithms](https://en.wikipedia.org/wiki/Genetic_algorithm), [Particle Swarm Optimization](https://en.wikipedia.org/wiki/Particle_swarm_optimization), and simulated annealing, which we develop below.

___

## Regularization: Preventing Overfitting
While gradient descent successfully minimizes the cross-entropy loss, the resulting model may fit the training data too closely, capturing noise and irrelevant patterns. As with ridge regression last week, we add a penalty on the size of the parameters. The regularized cross-entropy loss is given by:
$$
\boxed{
\begin{align*}
J_{\delta}(\mathbf{\theta}) & = J(\mathbf{\theta}) + \frac{\delta}{2}\,\lVert\mathbf{\theta}\rVert_{2}^{2}\\
& = \sum_{i=1}^n \log\!\bigl(1+\exp\bigl(-2\beta y_i\,\left(\hat{\mathbf{x}}^{\top}_{i}\mathbf{\theta}\right)\bigr)\bigr) + \frac{\delta}{2}\,\lVert\mathbf{\theta}\rVert_{2}^{2}\quad\blacksquare
\end{align*}}
$$
where $\delta\geq 0$ is the __regularization parameter__, which controls the strength of the penalty. The penalty adds $\delta\,\mathbf{\theta}$ to the gradient, $\nabla J_{\delta}(\mathbf{\theta}) = \nabla J(\mathbf{\theta}) + \delta\,\mathbf{\theta}$. Unlike ridge regression, there is no analytical solution, so we minimize $J_{\delta}(\mathbf{\theta})$ with gradient descent.

The penalty also helps gradient descent. With $\delta>0$, the regularized loss is __strongly convex__: it curves upward at least as much as the penalty does, everywhere. Thus, it has exactly one minimizer, even when the training data are linearly separable, and each step with a learning rate $\alpha\leq 1/(L+\delta)$ reduces the remaining error by at least a constant fraction.

> __In practice:__
>
> * __Scale the features__ to comparable magnitudes, e.g., zero mean and unit variance computed from the training data, so the penalty treats the parameters fairly.
> * __Choose $\delta$ by cross-validation__, as we did for ridge regression: compare candidate values by the error on data held out from fitting.

___

## Constrained Problems and Simulated Annealing
The logistic regression problem places no restrictions on $\mathbf{\theta}$, but many engineering problems restrict their decision variables, for example, to fractions that must be nonnegative and sum to one. Suppose we want to minimize a nonlinear objective function $f(x)$ of the decision variables $x\in\mathbb{R}^{n}$, subject to $m$ inequality constraints $g_i(x) \leq 0$ and $p$ equality constraints $h_j(x) = 0$:
$$
\begin{align*}
    \min_{x\in\mathbb{R}^n} \; f(x)
    \quad\text{s.t.}\quad
    \begin{cases}
    g_i(x) \le 0, & i = 1,\dots,m,\\
    h_j(x) = 0, & j = 1,\dots,p,
    \end{cases}
\end{align*}
$$
The [advanced optimization notebook](CHEME-5800-L9c-Advanced-NonlinearOptimization-Fall-2026.ipynb) develops the conditions that a solution of this problem must satisfy (the Karush–Kuhn–Tucker conditions) and a gradient descent method for it.

### Penalty and Barrier Terms
Simulated annealing, like gradient descent, works on an objective without constraints, so we fold the constraints into the objective:

> __Penalty and barrier methods:__ A __penalty__ term grows with the size of a constraint violation; we use a quadratic penalty for each equality constraint. A __barrier__ term grows without bound as $x$ approaches the edge of the region where the inequality constraints hold; we use a logarithmic barrier, which keeps the solution strictly inside that region.

Consider the following augmented objective function:
$$
\begin{align*}
    \min_{x\in\mathbb{R}^n}\;P_{\mu,\rho}(x)\;&=f(x)\;-\;\underbrace{\mu\sum_{i=1}^m\ln\bigl(-\,g_i(x)\bigr)}_{\text{barrier term}}\;+\;\underbrace{\frac{1}{2\rho}\sum_{j=1}^p
    \bigl[h_j(x)\bigr]^2}_{\text{penalty term}},\quad\text{where}\quad\mu>0,\;\rho>0
\end{align*}
$$

> __Parameters__
>
> The parameters $\mu$ and $\rho$ control the strength of these terms, and both are decreased over iterations:
> * __Barrier weight__: As $\mu\to0$, the barrier term fades inside the region, so the solution can approach its edge, where the optimum often lies. The barrier still grows without bound at the edge itself.
> * __Penalty weight__: As $\rho\to0$, the coefficient $\frac{1}{\rho}$ increases, strengthening the penalty effect and enforcing the equality constraints more strictly.

### Simulated Annealing
Simulated annealing (SA) is a derivative-free optimization method inspired by the annealing process in metallurgy (heating a metal and cooling it slowly so its atoms settle into a low-energy arrangement), developed by [Kirkpatrick, Gelatt, and Vecchi (1983)](https://doi.org/10.1126/science.220.4598.671). It explores the solution space by accepting both improving and non-improving moves, controlled by a temperature parameter. This allows the algorithm to avoid getting stuck in local optima, such as $\theta_{2}$ in the gradient descent figure.

__Why does this work?__ A move that raises the objective by $\Delta P>0$ is accepted with probability $\exp(-\Delta P/T)$, the Boltzmann factor from our derivation with $\beta=1/T$. At a high temperature, uphill moves are often accepted; as $T$ decreases, they become rare.

__Initialize__: Given an initial solution $x_0$ with $g_i(x_0)<0$ for all $i$, penalty parameters $\mu,\rho > 0$ with update factors $\tau_{\mu},\tau_{\rho}\in(0,1)$, an initial temperature $T\gets{T_\circ}$ and a minimum temperature $T_{\text{min}}$ with $T_{\circ}>T_{\text{min}}>0$, a cooling rate $\alpha\in(0,1)$, a step size $\eta > 0$, and the number of iterations $K$ per temperature. Set $\texttt{converged} \gets \texttt{false}$, the best solution $x^{\star} \gets x_0$, and the current solution $x_{c}\gets{x}_{0}$.

While not $\texttt{converged}$ __do__:

1. For $k = 1\,\text{to}\,K$ __do__:
   - Generate a candidate solution: $x^{\prime} \gets x_{c} + \eta\cdot\texttt{randn}(\texttt{size}(x_{c}))$, where $\texttt{randn}$ draws independent normal random numbers with mean zero and variance one.
   - If $g_i(x^{\prime})\geq 0$ for any $i$ __then__ reject the candidate and go to the next $k$; the barrier term is undefined there.
   - Compute the change in objective value: $\Delta P \gets P_{\mu,\rho}(x^{\prime}) - P_{\mu,\rho}(x_{c})$.
      - If $\Delta P < 0$ __then__ accept the new solution: $x_{c} \gets x^{\prime}$ (improving move).
      - If $\Delta P \geq 0$ __then__ accept with probability $p_{\text{accept}}\gets\exp(-\Delta P / T)$: draw $u \sim \texttt{Uniform}(0,1)$, and if $u \leq p_{\text{accept}}$, set $x_c \gets x^{\prime}$ (uphill move).
   - If $P_{\mu,\rho}(x_c) < P_{\mu,\rho}(x^{\star})$ __then__ update the best solution: $x^{\star}\gets{x}_{c}$. Both values use the current $\mu$ and $\rho$.
2. Update penalty parameters: $\mu\gets \tau_\mu\,\mu$ and $\rho\gets \tau_\rho\,\rho$.
3. Check the stopping condition:
   - If $T \leq T_{\text{min}}$ __then__ the temperature schedule is complete. Set $\texttt{converged} \gets \texttt{true}$ and return $x^{\star}$.
   - Otherwise, update temperature: $T \gets \alpha T$.

The equality constraints enter only through the penalty term, so we check how close $h_j(x^{\star})$ is to zero for each $j$ at the returned solution.

### Selecting $T_{\circ}$ and $K$
Simulated annealing is a heuristic, so it does not guarantee the global minimum, and its result depends on the temperature schedule. Two practical rules help us choose $T_{\circ}$ and $K$:

#### Sample-and-set approach for $T_\circ$
A practical way to select the initial temperature $T_\circ$ is to choose how often an uphill move should be accepted at the start:
1. From the initial solution $x_0$, generate $M=100$ random candidate solutions as in step 1 of the algorithm. Discard any candidate that violates an inequality constraint, and compute the changes in objective value $\Delta{P}_{j}$ for the rest.
2. Find the uphill moves: $\mathcal{U}\gets\left\{j: \Delta{P}_{j}>0\right\}$. If $\mathcal{U}$ is empty, generate more candidates. Compute the mean uphill change $\overline{\Delta P}_{+}\gets\texttt{mean}\left\{\Delta{P}_{j}\right\}_{j\in\mathcal{U}}$.
3. Choose a desired initial acceptance rate $p_{\circ}\in(0.6,0.9)$.
4. Set $T_{\circ}\gets{-{\overline{\Delta P}_{+}}/{\,\ln{p_{\circ}}}}$, so that an average uphill move is accepted with probability $\exp\left(-\overline{\Delta P}_{+}/T_{\circ}\right)=p_{\circ}$ at the start.

#### Heuristic for choosing $K$
Adapt the number of iterations per temperature based on acceptance rates:
1. Initially, set $K\gets\lceil c\,n\rceil$, where $n$ is the number of decision variables and $c\in\left[10,50\right]$ is the number of iterations per decision variable.
2. After each temperature, compute the acceptance rate $\hat{p}$ (fraction of accepted moves).
    - If $\hat{p}>0.8$, decrease $K\gets\lceil 0.75K\rceil$.
    - If $\hat{p}<{0.2}$, increase $K\gets\lceil 1.5K\rceil$.

Let's consider an example that uses both optimization methods.

> __Example__
>
> [▶ Fit a logistic regression classifier with gradient descent and simulated annealing](CHEME-5800-L9c-Example-LogisticRegression-GD-SA-Fall-2026.ipynb). Fit a logistic regression model to two overlapping clusters with gradient descent and with simulated annealing, and classify test points at several thresholds.

___

## Lab
In the L9d lab, we'll train a logistic regression classifier with gradient descent on the [banknote authentication dataset](https://archive.ics.uci.edu/dataset/267/banknote+authentication), which distinguishes genuine from forged banknotes using features extracted from images of the notes. We'll classify the test examples, use a confusion matrix to evaluate the classifier, and compare it with the Perceptron on the same banknotes. We'll also use its probabilities to decide which banknotes a person should check by hand.

___

## Summary
In this lecture, we derived logistic regression from the Boltzmann distribution, estimated its parameters with gradient descent, and solved constrained problems with simulated annealing.

> __Key Takeaways:__
>
> * __The logistic regression model:__ We treated the two class labels as states in a Boltzmann distribution with an energy that is linear in the features, and derived the logistic function for the probability of each label. Predicting the more probable label gives the Perceptron's decision boundary, and the probability tells us how confident the model is.
>
> * __Parameter estimation with gradient descent:__ We estimated the parameters by minimizing the cross-entropy loss with gradient descent, where each step weights every example by the probability the model gives to its wrong label. Adding a ridge penalty gives the loss a unique minimizer, even when the classes are linearly separable.
>
> * __Constrained problems and simulated annealing:__ We folded constraints into the objective with penalty and barrier terms and minimized the result with simulated annealing. Accepting some uphill moves, more often at high temperature, lets the search climb out of local minima that can trap gradient descent.

Next week, we'll look at online learning, where a model updates its parameters as each new example arrives.

___